In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config.settings_2011 import *

from src.io_utils import load_dsn_data, load_horizons_daily_sep
from src.doppler_utils import prepare_daily_rms_table, print_daily_summary
from src.phase_utils import compute_phase_rms_windows, print_phase_summary

In [3]:
from config.settings_2010 import *

from src.io_utils import load_dsn_data, load_horizons_daily_sep
from src.doppler_utils import prepare_daily_rms_table, print_daily_summary
from src.phase_utils import compute_phase_rms_windows, print_phase_summary

# Load
horizons_daily = load_horizons_daily_sep(HORIZONS_FILE)
df = load_dsn_data(
    DOPPLER_FILE,
    valid_only=True,
    min_elev_deg=MIN_ELEV_DEG,
    max_abs_doppler_hz=MAX_ABS_DOPPLER_HZ,
)

print("Horizons daily rows:", len(horizons_daily))
print("Horizons time range:", horizons_daily["day"].min(), "→", horizons_daily["day"].max())
print("DSN rows after filtering:", len(df))
print("DSN time range:", df["UTC_time"].min(), "→", df["UTC_time"].max())

# Daily RMS
daily_df = prepare_daily_rms_table(
    dsn_df=df,
    horizons_daily=horizons_daily,
    f_carrier_hz=F_CARRIER,
    c_mps=C,
    T_int_sec=T_INT,
    C_band=C_BAND,
    resample_rule="60s",
    min_samples_per_day=MIN_SAMPLES_PER_DAY,
    smooth_days=SMOOTH_DAYS,
    add_tropo_diagnostic=True,
)
print_daily_summary(daily_df)

# Phase windows
windows_df = compute_phase_rms_windows(
    df,
    dt_target_sec=DT_TARGET,
    window_min=WINDOW_MIN,
    step_min=STEP_MIN,
    min_samples=MIN_SAMPLES,
    f_low_hz=F_LOW,
    f_high_hz=F_HIGH,
)
print_phase_summary(windows_df)

Horizons daily rows: 363
Horizons time range: 2010-01-01 00:00:00 → 2010-12-29 00:00:00
DSN rows after filtering: 867761
DSN time range: 2010-01-01 09:37:22.814000 → 2010-12-30 12:39:54.533000

Daily RMS rows: 336
Daily RMS time range: 2010-01-01 00:00:00 → 2010-12-30 00:00:00
Measured Doppler RMS range (mm/s): 0.024075211890810536 → 3.876641920322963
Missing SEP after daily merge: 1
SEP range (deg): 1.0374 → 46.5264
Windows created: 14590
Window time range: 2010-01-01 09:37:20 → 2010-12-30 12:37:20
Phase RMS range (rad): 0.04816102296569241 → 63.824174239744025
